- Before we can do vector search, we need to turn our text into vectors. 
- We call this process embedding: we embed text into a vector space. 
- The vectors we get back are also called "embeddings."
- So an embedding model takes text in and returns a fixed-length array of numbers. 
- We train it so that texts with similar meanings get similar vectors.
- We'll use `sentence-transformers`, a popular open-source library for embeddings.

### Imports

In [ ]:
from sentence_transformers import SentenceTransformer
from ingest import load_faq_data, build_index
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from google import genai
import os
from sqlitesearch import VectorSearchIndex


load_dotenv()

True

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1) # v1 is a vector, an array of 384 numbers.
print(len(v1))
v1[:10]

384


array([ 0.02139038, -0.07397996,  0.00142068,  0.02138166,  0.02451134,
        0.0315583 , -0.11083971, -0.10501751, -0.0618259 , -0.00642313],
      dtype=float32)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
print(len(dv))
dv[:10]

384


array([-0.0481207 , -0.12994204, -0.00486809,  0.01374952, -0.008511  ,
        0.06978405, -0.00755664,  0.04095721, -0.10216569, -0.0345051 ],
      dtype=float32)

In [5]:
# we compare the query against the document using dot product:
v1.dot(dv)

np.float32(0.32332397)

In [6]:
# Now we try an unrelated query:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
print(len(v2))
v2[:10]

384


array([-0.02356785,  0.07855561,  0.02337924, -0.00672473, -0.02588749,
        0.01182574,  0.00052957,  0.00957371,  0.06239291,  0.02914272],
      dtype=float32)

In [7]:
# This time the similarity with the document should be much smaller:
v2.dot(dv)

np.float32(0.019730518)

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

### Cosine similarity
- The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. 
- When both vectors are normalized, the dot product equals cosine similarity. 
- Cosine similarity measures the angle between two vectors, ignoring their length:
    - 1.0 = same direction (similar)
    - 0.0 = perpendicular (unrelated)
    - 1.0 = opposite direction (opposite meaning)

### Embedding Our Dataset

In [8]:
documents = load_faq_data()
print(f"Type: {type(documents)}; Count: {len(documents)}")
documents[0]

Type: <class 'list'>; Count: 1380


{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

#### Generating embeddings

In [9]:
# Each document is a Python dictionary with a question and an answer. 
# We embed both together. 
# That way a query can match against the question text and the answer text in our index.
# Build one text per document:

texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

texts[0]

"How do I submit homework? - Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"

In [10]:
# Now we generate the embeddings. 
# We have about 1350 texts here.
# We won't hand the model all of them at once. 
# That takes a long time, and we can't see what's happening inside. 
# Instead we split them into batches.
# import tqdm so we can watch the progress:
# Next we chunk the dataset into batches of 50 and encode each batch:

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

print(f"Count: {len(vectors)}")
vectors[0]

  0%|          | 0/28 [00:00<?, ?it/s]

Count: 1380


array([-1.03578992e-01, -6.19631894e-02, -1.13746114e-02, -3.32002225e-03,
        2.98824627e-02, -1.73465721e-02, -9.54300314e-02,  3.11984047e-02,
       -1.53322116e-01,  7.39517137e-02,  4.03770618e-02, -4.62293625e-02,
        1.96900908e-02, -3.71884219e-02, -1.84310488e-02,  2.22526919e-02,
        5.88778686e-03, -5.50784245e-02,  5.64357825e-02, -1.05830461e-01,
        4.25289124e-02, -3.97676528e-02, -1.17405998e-02,  7.39043877e-02,
        4.40801494e-02,  5.91114163e-02,  6.24631085e-02, -2.63405964e-02,
       -8.60163290e-03,  5.82083166e-02, -1.20065985e-02, -1.86079927e-03,
        2.75744870e-02,  3.93734574e-02,  6.54970855e-02,  6.49855807e-02,
        1.75080039e-02, -4.75835986e-02, -7.16399252e-02, -6.99092150e-02,
       -6.10002540e-02, -1.18092569e-02,  4.61779797e-04,  4.62616868e-02,
        1.83203612e-02, -8.85390565e-02, -3.84005420e-02, -9.99948829e-02,
       -4.70628701e-02,  4.94977124e-02, -4.06513102e-02, -1.30986705e-01,
       -6.99002668e-02,  

In [11]:
# We turn them into a 2-dimensional array (matrix) where
# rows are documents (vectors)
# columns are dimensions of the vectors

X = np.array(vectors)
print(f"Count: {X.shape}")
X[0][:5]

Count: (1380, 384)


array([-0.10357899, -0.06196319, -0.01137461, -0.00332002,  0.02988246],
      dtype=float32)

### Vector Search

#### Scoring documents

In [12]:
# We have a matrix X with all document embeddings. 
# We take a query, compare it against every document, and keep the most similar ones.
# When a query comes in, we embed it:

query = "Can I still join the course after the start date?"
v_query = model.encode(query)
v_query[:5]

array([ 0.02139038, -0.07397996,  0.00142068,  0.02138166,  0.02451134],
      dtype=float32)

In [13]:
# Next, we compute the dot product against all documents:
# This is matrix-vector multiplication. 
# Each element i of scores is the cosine similarity between document i (row i of X) and v_query
scores = X.dot(v_query)
print(scores.shape)
scores[:5]

(1380,)


array([0.23396146, 0.05057279, 0.00561458, 0.30945054, 0.10449579],
      dtype=float32)

In [14]:
# The highest score is the most similar document:
idx = np.argmax(scores)
print(f"Best match index: {idx}; Score: {scores[idx]}")
documents[idx]

Best match index: 860; Score: 0.7629410624504089


{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [15]:
# top 5
# np.argsort sorts from lowest to highest, so the last 5 are the top ones:
top5 = np.argsort(scores)[-5:]
top5

array([ 865, 1262,   29,  473,  860])

In [16]:
# They come out smallest-first, so we reverse them to get the highest first:
top5 = top5[::-1]
top5

array([ 860,  473,   29, 1262,  865])

In [17]:
# There's a shorter trick
# We negate the scores first, so the largest becomes the smallest. 
# Then argsort puts the best matches at the front.

top5 = np.argsort(-scores)[:5]
top5

array([ 860,  473,   29, 1262,  865])

In [18]:
scores[top5]

array([0.76294106, 0.7579371 , 0.71921325, 0.65363115, 0.56009984],
      dtype=float32)

In [19]:
for idx in top5:
    print(f"index: {idx}; Score: {scores[idx]}")
    print(documents[idx])
    print()

index: 860; Score: 0.7629410624504089
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

index: 473; Score: 0.7579370737075806
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

index: 29; Score: 0.7192132472991943
{'id': '41a

This is vector search in its simplest form. 
We embed the query, compute dot products against all documents, and return the highest-scoring ones.

### Vector Search with minsearch

In [20]:
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X,documents)

In [21]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

#### Filtering by course

In [22]:
results = vindex.search(
    query_vector=query_vector,
    filter_dict={"course":"llm-zoomcamp"},
    num_results=5
)
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Now that we can run vector search, let's use it in RAG.

### RAG with Vector Search

#### Using RAGBase

In [23]:
# First, create the OpenAI client:

# openai_client = OpenAI()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# client = genai.Client(api_key=GOOGLE_API_KEY)

# Point the OpenAI client at Google's Gemini OpenAI-compatible endpoint
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [24]:
# Next, download and index the data:
documents = load_faq_data()
index = build_index(documents)

In [25]:
# Then use the RAGBase class:

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
)

In [27]:
# Ask it a question:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

### Vector Search with sqlitesearch

In the previous section we used minsearch for vector search.

It works, but it has three problems:
- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force

sqlitesearch is the persistent sibling of minsearch, and it solves both problems at once.
- It also does vector search through its VectorSearchIndex 

#### Creating the index

In [29]:
# Initialize it:

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch supports three ANN modes:

- lsh (default): up to 100K vectors, random hyperplane projections
- ivf: 10K-500K vectors, K-means clustering
- hnsw: 10K-1M+ vectors, proximity graph (highest recall)

For our small dataset, lsh is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

#### Indexing the data

In [30]:
# Fit the index with our vectors and documents:
vs_index.fit(vectors, documents)
# The index is saved to faq_vectors2.db. Unlike minsearch, this file persists on disk. 
# You can search immediately after indexing, or reopen the index later without re-indexing.

#### Searching

In [ ]:
# 
# Search works the same way as with minsearch. 
# We always encode the query into a vector first. 
# This is one thing that makes vector search heavier than text search. 
# With text search we'd throw the raw query straight at the engine.
# Encode, then search:

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
results
# 

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

#### Filtering by course

In [36]:
# Filtering by course
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

#### Closing the connection

In [40]:
# When you're done with the index:
vs_index.close()

#### Reopening the index

In [41]:
# In a new Python session, you can reopen the index without re-computing embeddings:
model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
# Now we can search:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

### Using sqlitesearch vector search in RAG